# exp_e2_mpnet_multi — Semantic Graph Builder v2

**Phase 1a, embedder = `paraphrase-multilingual-mpnet-base-v2`, LLM = `deepseek-v32/latest` (API).**

Запускается из `exps/FINAL_EXPS/phase1a_embedders/exp_e2_mpnet_multi/` — все пути разрешаются автоматически от `clustering_1/`.

Перед запуском: `export YANDEX_CLOUD_API_KEY=...`.

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import torch

In [2]:
import os, sys, logging
from pathlib import Path

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')

# layout: clustering_1/exps/FINAL_EXPS/phase1a_embedders/exp_e2_mpnet_multi/
EXP_DIR   = Path().resolve()
REPO_ROOT = EXP_DIR.parents[3]                 # clustering_1/
LLM_V2    = REPO_ROOT / 'llm_v2'

# put repo root on sys.path so `import llm_v2` works
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
# guard: never expose llm_v2/ as flat path
if str(LLM_V2) in sys.path:
    sys.path.remove(str(LLM_V2))

from llm_v2.config_schema import load_config
config = load_config(EXP_DIR / 'config.yaml')

# expand ${YANDEX_CLOUD_API_KEY} etc. (no-op for local LLMs)
config.llm.api_key = os.path.expandvars(config.llm.api_key)
config.llm.base_url = os.path.expandvars(config.llm.base_url)
config.llm.folder = os.path.expandvars(config.llm.folder)

print('EXP_DIR  :', EXP_DIR)
print('REPO_ROOT:', REPO_ROOT)
print('LLM_V2   :', LLM_V2)
print()
print(config.model_dump_json(indent=2))

EXP_DIR  : /home/platoon/graph/semantic-graph/exps/FINAL_EXPS/phase1a_embedders/exp_e3_e5_large
REPO_ROOT: /home/platoon/graph/semantic-graph
LLM_V2   : /home/platoon/graph/semantic-graph/llm_v2

{
  "llm": {
    "provider": "api",
    "model_name": "deepseek-v32/latest",
    "max_new_tokens": 500,
    "temperature": 0.3,
    "device": "cpu",
    "load_in_8bit": false,
    "api_key": "${YANDEX_CLOUD_API_KEY}",
    "base_url": "https://ai.api.cloud.yandex.net/v1",
    "folder": "b1gpiug3vgbpe1cb4e5c",
    "instructions": ""
  },
  "embedding": {
    "model_name": "intfloat/multilingual-e5-large",
    "device": "cuda"
  },
  "coreference": {
    "enabled": false,
    "prompt_file": "prompts/coreference_ru.txt",
    "context_sentences": 3,
    "window_sentences": 5
  },
  "extraction": {
    "prompt_file": "prompts/extraction_ru.txt",
    "chunk_size": 3,
    "overlap_size": 1
  },
  "normalization": {
    "enabled": true,
    "language": "ru"
  },
  "deduplication": {
    "enabled": true

In [ ]:
config.llm.api_key = ""

In [4]:
from llm_v2.models.llm_client import LLMClient
from llm_v2.models.embedder import Embedder

llm = LLMClient(config.llm)
embedder = Embedder(config.embedding)
print(f'LLM loaded: {config.llm.model_name}')
print(f'Embedder loaded: {config.embedding.model_name} (dim={embedder.dim})')

/home/platoon/graph/graph_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-05-07 15:16:32,314 [INFO] Load pretrained SentenceTransformer: intfloat/multilingual-e5-large
2026-05-07 15:16:32,659 [INFO] HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-large/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-05-07 15:16:32,839 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/intfloat/multilingual-e5-large/3d7cfbdacd47fdda877c5cd8a79fbcc4f2a574f3/modules.json "HTTP/1.1 200 OK"
2026-05-07 15:16:33,013 [INFO] HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-large/resolve/main/config_sentence_transformers.json "HTTP/1.1 404 Not Found"
2026-05-07 15:16:33,194 [INFO] HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e

LLM loaded: deepseek-v32/latest
Embedder loaded: intfloat/multilingual-e5-large (dim=1024)


In [5]:
from llm_v2.utils.io import load_text

input_path = Path(config.paths.input_text)
if not input_path.is_absolute():
    input_path = (LLM_V2 / input_path).resolve()
text = load_text(input_path)
print(f'Input: {input_path}')
print(f'Length: {len(text)} chars')
print(text[:500])

Input: /home/platoon/graph/semantic-graph/benchmark/final_bench/formated_fragment2.md
Length: 15466 chars
# Линейная классификация

Теперь давайте поговорим про задачу классификации. Для начала будем говорить про бинарную классификацию на два класса. Обобщить эту задачу до задачи классификации на $K$ классов не составит большого труда.

Пусть теперь наши таргеты $y$ кодируют принадлежность к положительному или отрицательному классу, то есть принадлежность множеству $\{-1,1\}$, а $x$ — по-прежнему векторы из $\mathbb{R}^D$.

В этом параграфе договоримся именно так обозначать классы, хотя в жизни вам 


## [0] Preprocessing

In [6]:
from llm_v2.stages.preprocessing import preprocess

sentences = preprocess(text, language=config.normalization.language)
for s in sentences:
    print(f'  [{s.id}] {s.text}')

  [0] # Линейная классификация

Теперь давайте поговорим про задачу классификации.
  [1] Для начала будем говорить про бинарную классификацию на два класса.
  [2] Обобщить эту задачу до задачи классификации на $K$ классов не составит большого труда.
  [3] Пусть теперь наши таргеты $y$ кодируют принадлежность к положительному или отрицательному классу, то есть принадлежность множеству $\{-1,1\}$, а $x$ — по-прежнему векторы из $\mathbb{R}^D$.
  [4] В этом параграфе договоримся именно так обозначать классы, хотя в жизни вам будут нередко встречаться и метки $\{0,1\}$.
  [5] Мы хотим обучить линейную модель так, чтобы плоскость, которую она задаёт, как можно лучше отделяла объекты одного класса от объектов другого.
  [6] **2.1.6**

В идеальной ситуации найдётся плоскость, которая разделит классы: положительный окажется с одной стороны от неё, а отрицательный — с другой.
  [7] Выборка, для которой это возможно, называется линейно разделимой.
  [8] Увы, в реальной жизни такое встречается кр

## [1] Coreference Resolution

In [7]:
from llm_v2.stages.coreference import resolve_coreferences

resolved_text, sentences = resolve_coreferences(
    sentences, llm, config.coreference, base_dir=LLM_V2
)
print('Resolved text:')
print(resolved_text)
print(f'\nSentences after coref: {len(sentences)}')

Resolved text:
# Линейная классификация

Теперь давайте поговорим про задачу классификации. Для начала будем говорить про бинарную классификацию на два класса. Обобщить эту задачу до задачи классификации на $K$ классов не составит большого труда. Пусть теперь наши таргеты $y$ кодируют принадлежность к положительному или отрицательному классу, то есть принадлежность множеству $\{-1,1\}$, а $x$ — по-прежнему векторы из $\mathbb{R}^D$. В этом параграфе договоримся именно так обозначать классы, хотя в жизни вам будут нередко встречаться и метки $\{0,1\}$. Мы хотим обучить линейную модель так, чтобы плоскость, которую она задаёт, как можно лучше отделяла объекты одного класса от объектов другого. **2.1.6**

В идеальной ситуации найдётся плоскость, которая разделит классы: положительный окажется с одной стороны от неё, а отрицательный — с другой. Выборка, для которой это возможно, называется линейно разделимой. Увы, в реальной жизни такое встречается крайне редко. Как обучить линейную модель

## [1.5] Chunking

In [8]:
from llm_v2.stages.chunking import build_chunks

chunks = build_chunks(sentences, config.extraction)
for c in chunks:
    print(f'  {c.id} (sents {c.sentence_ids}): {c.text[:80]}...')

  chunk_0 (sents [0, 1, 2]): # Линейная классификация

Теперь давайте поговорим про задачу классификации. Для...
  chunk_1 (sents [2, 3, 4]): Обобщить эту задачу до задачи классификации на $K$ классов не составит большого ...
  chunk_2 (sents [4, 5, 6]): В этом параграфе договоримся именно так обозначать классы, хотя в жизни вам буду...
  chunk_3 (sents [6, 7, 8]): **2.1.6**

В идеальной ситуации найдётся плоскость, которая разделит классы: пол...
  chunk_4 (sents [8, 9, 10]): Увы, в реальной жизни такое встречается крайне редко. Как обучить линейную модел...
  chunk_5 (sents [10, 11, 12]): $$

<details>
<summary>Почему бы не решать задачу классификации как задачу регре...
  chunk_6 (sents [12, 13, 14]): Во вторых, ошибкой будет считаться предсказание, например, $5$ вместо $1$, хотя ...
  chunk_7 (sents [14, 15, 16]): </details>

Сконструируем теперь функционал ошибки так, чтобы он вышеперечисленн...
  chunk_8 (sents [16, 17, 18]): $$

Домножим обе части на $y_i$ и немного упростим:

$

## [2] Triplet Extraction

In [9]:
from llm_v2.stages.extraction import extract_triplets

raw_triplets = extract_triplets(chunks, llm, config.extraction, base_dir=LLM_V2)
print(f'Extracted {len(raw_triplets)} raw triplets:')
for t in raw_triplets:
    print(f'  {t.subject} | {t.relation} | {t.object}  [{t.chunk_id}]')

Extracting triplets: 100%|██████████| 56/56 [10:24<00:00, 11.14s/it]

Extracted 371 raw triplets:
  линейная классификация | является темой разговора | задача классификации  [chunk_0]
  задача классификации | начинается с | бинарная классификация  [chunk_0]
  бинарная классификация | имеет количество классов | два класса  [chunk_0]
  задача классификации | может быть обобщена до | классификация на K классов  [chunk_0]
  задача классификации | обобщается до | K классов  [chunk_1]
  таргеты y | кодируют | принадлежность к положительному классу  [chunk_1]
  таргеты y | кодируют | принадлежность к отрицательному классу  [chunk_1]
  принадлежность | является принадлежностью множеству | {-1,1}  [chunk_1]
  x | является | векторы  [chunk_1]
  векторы | принадлежат | ℝ^D  [chunk_1]
  классы | обозначаются | {-1,1}  [chunk_1]
  метки | встречаются | {0,1}  [chunk_1]
  мы | договорились обозначать | классы  [chunk_2]
  мы | хотим обучить | линейная модель  [chunk_2]
  линейная модель | задаёт | плоскость  [chunk_2]
  плоскость | должна отделять | объекты одного кл

## [3] Normalization

In [10]:
from llm_v2.stages.normalization import normalize_triplets

norm_triplets = normalize_triplets(raw_triplets, config.normalization)
print(f'Normalized {len(norm_triplets)} triplets:')
for t in norm_triplets:
    print(f'  {t.norm_subject} | {t.norm_relation} | {t.norm_object}')

2026-05-07 15:27:56,605 [INFO] Loading dictionaries from /home/platoon/graph/graph_env/lib/python3.12/site-packages/pymorphy3_dicts_ru/data
2026-05-07 15:27:56,635 [INFO] format: 2.4, revision: 417150, updated: 2022-01-08T22:09:24.565962


Normalized 371 triplets:
  линейный классификация | являться тема разговор | задача классификация
  задача классификация | начинаться с | бинарный классификация
  бинарный классификация | иметь количество класс | два класс
  задача классификация | мочь быть обобщить до | классификация на k класс
  задача классификация | обобщаться до | k класс
  таргет y | кодировать | принадлежность к положительный класс
  таргет y | кодировать | принадлежность к отрицательный класс
  принадлежность | являться принадлежность множество | {-1,1}
  x | являться | вектор
  вектор | принадлежать | ℝ^D
  класс | обозначаться | {-1,1}
  метка | встречаться | {0,1}
  мы | договориться обозначать | класс
  мы | хотеть обучить | линейный модель
  линейный модель | задавать | плоскость
  плоскость | должный отделять | объект один класс
  плоскость | должный отделять | объект другой класс
  плоскость | мочь разделить | класс
  положительный класс | оказаться с один сторона | плоскость
  плоскость | разделять | кл

## [4] Deduplication

In [11]:
from llm_v2.stages.deduplication import deduplicate_triplets

dedup_triplets = deduplicate_triplets(norm_triplets, embedder, config.deduplication)
print(f'After dedup: {len(norm_triplets)} -> {len(dedup_triplets)} triplets')
for t in dedup_triplets:
    print(f'  {t.norm_subject} | {t.norm_relation} | {t.norm_object}')

After dedup: 371 -> 218 triplets
  задача классификация | начинаться с | бинарный классификация
  бинарный классификация | иметь количество класс | два класс
  задача классификация | обобщаться до | k класс
  таргет y | кодировать | принадлежность к положительный класс
  принадлежность | являться принадлежность множество | {-1,1}
  x | являться | вектор
  вектор | принадлежать | ℝ^D
  класс | обозначаться | {-1,1}
  метка | встречаться | {0,1}
  мы | договориться обозначать | класс
  мы | хотеть обучить | линейный модель
  линейный модель | задавать | плоскость
  положительный класс | оказаться с один сторона | плоскость
  плоскость | разделять | класс
  выборка | называться | линейно разделимый выборка
  линейно разделимый выборка | возможный при | разделение класс плоскость
  линейно разделимый выборка | встречаться редко | в реальный жизнь
  регрессия | мочь предсказывать | число -1 и 1
  предсказание число | минимизировать | MSE
  предсказание число | включать последующий взятие | 

## [5] Graph Assembly (raw)

In [12]:
from llm_v2.stages.graph_assembly import assemble_graph

raw_graph = assemble_graph(dedup_triplets, chunks, text, config)
print(f'Raw graph: {len(raw_graph.nodes)} nodes, {len(raw_graph.edges)} edges')
print('\nNodes:')
for n in raw_graph.nodes:
    print(f'  {n.id}: {n.label} ({len(n.mentions)} mentions)')
print('\nEdges:')
for e in raw_graph.edges:
    print(f'  {e.id}: {e.source} --[{e.label}]--> {e.target} (w={e.weight})')

Raw graph: 280 nodes, 218 edges

Nodes:
  n0: задача классификация (2 mentions)
  n1: бинарный классификация (2 mentions)
  n2: два класс (1 mentions)
  n3: k класс (1 mentions)
  n4: таргет y (1 mentions)
  n5: принадлежность к положительный класс (1 mentions)
  n6: принадлежность (1 mentions)
  n7: {-1,1} (2 mentions)
  n8: x (1 mentions)
  n9: вектор (2 mentions)
  n10: ℝ^D (1 mentions)
  n11: класс (6 mentions)
  n12: метка (1 mentions)
  n13: {0,1} (1 mentions)
  n14: мы (15 mentions)
  n15: линейный модель (4 mentions)
  n16: плоскость (3 mentions)
  n17: положительный класс (2 mentions)
  n18: выборка (1 mentions)
  n19: линейно разделимый выборка (3 mentions)
  n20: разделение класс плоскость (1 mentions)
  n21: в реальный жизнь (1 mentions)
  n22: регрессия (2 mentions)
  n23: число -1 и 1 (1 mentions)
  n24: предсказание число (2 mentions)
  n25: MSE (1 mentions)
  n26: знак (3 mentions)
  n27: ошибка на объект (2 mentions)
  n28: разделять плоскость (2 mentions)
  n29: предс

## [6] Clustering

In [13]:
from llm_v2.stages.clustering import cluster_graph, cluster_graph_multi, cluster_graph_all_methods
from llm_v2.utils.io import load_prompt

naming_prompt_path = Path(config.clustering.cluster_naming_prompt)
if not naming_prompt_path.is_absolute():
    naming_prompt_path = (LLM_V2 / naming_prompt_path).resolve()
naming_prompt = load_prompt(naming_prompt_path) if naming_prompt_path.exists() else None

if config.clustering.multi_method:
    multi = cluster_graph_all_methods(
        raw_graph, embedder, config,
        llm=llm, prompt_template=naming_prompt,
    )
    print('Multi-method clustering:')
    for method_name, mr in multi.methods.items():
        print(f'  {method_name}: {len(mr.param_labels)} variants')
        for lbl in mr.param_labels:
            g = mr.graphs[lbl]
            print(f'    {lbl}: {len(g.nodes)} nodes, {len(g.edges)} edges')
    agg = multi.methods['agglomerative']
    mid_label = agg.param_labels[len(agg.param_labels) // 2]
    clustered = agg.graphs[mid_label]
elif config.clustering.is_multi_threshold:
    multi = cluster_graph_multi(
        raw_graph, embedder, config,
        llm=llm, prompt_template=naming_prompt,
    )
    agg = multi.methods['agglomerative']
    print(f'Multi-threshold: {len(agg.param_labels)} levels')
    for lbl in agg.param_labels:
        g = agg.graphs[lbl]
        print(f'  t={lbl}: {len(g.nodes)} nodes, {len(g.edges)} edges')
    mid_label = agg.param_labels[len(agg.param_labels) // 2]
    clustered = agg.graphs[mid_label]
else:
    clustered = cluster_graph(
        raw_graph, embedder, config,
        llm=llm, prompt_template=naming_prompt,
    )

print(f'\nClustered graph: {len(clustered.nodes)} nodes, {len(clustered.edges)} edges')
print('\nClustered Nodes:')
for n in clustered.nodes:
    print(f'  {n.id}: {n.label} (members={n.members}, size={n.size})')
print('\nClustered Edges:')
for e in clustered.edges:
    print(f'  {e.id}: {e.source} --[{e.label}]--> {e.target} (size={e.size})')

Multi-method clustering:
  agglomerative: 10 variants
    0.250: 71 nodes, 81 edges
    0.322: 47 nodes, 58 edges
    0.394: 41 nodes, 53 edges
    0.467: 26 nodes, 44 edges
    0.539: 14 nodes, 21 edges
    0.611: 2 nodes, 2 edges
    0.683: 1 nodes, 0 edges
    0.756: 1 nodes, 0 edges
    0.828: 1 nodes, 0 edges
    0.900: 1 nodes, 0 edges
  kmeans: 4 variants
    k=10: 10 nodes, 25 edges
    k=25: 25 nodes, 44 edges
    k=40: 40 nodes, 53 edges
    k=55: 55 nodes, 64 edges
  hdbscan: 9 variants
    mcs=3,ms=1: 55 nodes, 64 edges
    mcs=3,ms=3: 60 nodes, 69 edges
    mcs=3,ms=5: 105 nodes, 101 edges
    mcs=5,ms=1: 72 nodes, 74 edges
    mcs=5,ms=3: 75 nodes, 77 edges
    mcs=5,ms=5: 107 nodes, 102 edges
    mcs=10,ms=1: 51 nodes, 36 edges
    mcs=10,ms=3: 57 nodes, 41 edges
    mcs=10,ms=5: 80 nodes, 60 edges

Clustered graph: 2 nodes, 2 edges

Clustered Nodes:
  c0: мы (members=['n0', 'n1', 'n2', 'n3', 'n4', 'n5', 'n6', 'n7', 'n8', 'n9', 'n10', 'n11', 'n12', 'n13', 'n14', 'n15', '

## Save outputs

In [14]:
from llm_v2.utils.io import save_json, save_text

out = EXP_DIR / config.paths.output_dir
out.mkdir(parents=True, exist_ok=True)

save_text(resolved_text, out / 'coreference_resolved.txt')
save_json(raw_graph.model_dump(), out / 'raw_graph.json')
save_json(clustered.model_dump(), out / 'clustered_graph.json')

if config.clustering.multi_method or config.clustering.is_multi_threshold:
    save_json(multi.model_dump(), out / 'multi_clustered_graph.json')
    method_counts = {m: len(r.param_labels) for m, r in multi.methods.items()}
    print(f'Saved multi_clustered_graph.json (methods: {method_counts})')

print(f'Saved to {out}/')

Saved multi_clustered_graph.json (methods: {'agglomerative': 10, 'kmeans': 4, 'hdbscan': 9})
Saved to /home/platoon/graph/semantic-graph/exps/FINAL_EXPS/phase1a_embedders/exp_e3_e5_large/output/


## Benchmark vs ground-truth graph

In [15]:
from llm_v2.benchmark import (
    evaluate_graph,
    evaluate_multi_graph,
    load_clustered_graph,
    print_metrics,
    print_multi_metrics,
    best_variant,
    show_node_alignments,
    show_edge_alignments,
    multi_metrics_to_dict,
)

# GT inputs (absolute, robust to CWD)
gt_graph_path = REPO_ROOT / 'benchmark' / 'final_bench' / 'graph_clustered.json'
gt_text_path  = REPO_ROOT / 'benchmark' / 'final_bench' / 'formated_fragment2.md'

gt_graph = load_clustered_graph(gt_graph_path)
gt_text  = gt_text_path.read_text(encoding='utf-8')

# embedding context: prefer the coreference-resolved text the pipeline saw
source_text = resolved_text if resolved_text else gt_text

print(f'GT  : {len(gt_graph.nodes)} nodes, {len(gt_graph.edges)} edges  ({gt_graph_path})')
print(f'Pred: {len(clustered.nodes)} nodes, {len(clustered.edges)} edges')
print(f'Context text: {len(source_text)} chars')

TAU_NODE = 0.6
TAU_EDGE = 0.6
BETA = 1.0
NODE_WEIGHT = 0.6
EDGE_WEIGHT = 0.4
NODE_WINDOW = 300
EDGE_WINDOW = 400
TOP_K = 10

GT  : 55 nodes, 51 edges  (/home/platoon/graph/semantic-graph/benchmark/final_bench/graph_clustered.json)
Pred: 2 nodes, 2 edges
Context text: 15418 chars


In [16]:
metrics = evaluate_graph(
    pred=clustered,
    gt=gt_graph,
    source_text=source_text,
    embedder=embedder,
    tau_node=TAU_NODE,
    tau_edge=TAU_EDGE,
    beta=BETA,
    node_weight=NODE_WEIGHT,
    edge_weight=EDGE_WEIGHT,
    node_window=NODE_WINDOW,
    edge_window=EDGE_WINDOW,
)

print_metrics(metrics)
metrics.summary()

Batches: 100%|██████████| 1/1 [00:00<00:00, 104.87it/s]


GraphScore = 0.6·NodeF1 + 0.4·EdgeF1  =  0.0655

Nodes (pred=2, gt=55, matched=2, tau=0.6, beta=1):
  TP(soft)  = 1.7986
  precision = 0.8993
  recall    = 0.0327
  F1        = 0.0631
Edges (pred=2, gt=51, matched=2, tau=0.6, beta=1):
  TP(soft)  = 1.8289
  precision = 0.9144
  recall    = 0.0359
  F1        = 0.0690


{'graph_score': 0.06547135309997831,
 'node_weight': 0.6,
 'edge_weight': 0.4,
 'nodes': {'precision': 0.899309366941452,
  'recall': 0.03270215879787098,
  'f_beta': 0.06310942925904926,
  'beta': 1.0,
  'tau': 0.6,
  'tp': 1.798618733882904,
  'pred_count': 2,
  'gt_count': 55,
  'matched_count': 2},
 'edges': {'precision': 0.9144386649131775,
  'recall': 0.03586033980051676,
  'f_beta': 0.06901423886137188,
  'beta': 1.0,
  'tau': 0.6,
  'tp': 1.828877329826355,
  'pred_count': 2,
  'gt_count': 51,
  'matched_count': 2},
 'pred_structure': {'n_nodes': 2,
  'n_edges': 2,
  'density': 1.0,
  'n_components': 1,
  'n_isolated': 0,
  'component_sizes': [2],
  'component_size_min': 2,
  'component_size_max': 2,
  'component_size_mean': 2.0,
  'component_size_quantiles': {'q25': 2.0,
   'q50': 2.0,
   'q75': 2.0,
   'q90': 2.0}},
 'gt_structure': {'n_nodes': 55,
  'n_edges': 51,
  'density': 0.01717171717171717,
  'n_components': 6,
  'n_isolated': 0,
  'component_sizes': [20, 11, 9, 7, 6,

In [17]:
show_node_alignments(clustered, gt_graph, metrics, top_k=TOP_K)

Matched node pairs: 2 / min(2, 55)=2

Top 2 matched (by quality q):
  [q=0.900]  'SVM'  ↔  'вектор'
  [q=0.898]  'мы'  ↔  'число ошибок классификатора'

Unmatched GT nodes (53):
  'задача классификации'
  'бинарная классификация'
  'классификация на $K$ классов'
  'таргет $y$'
  'положительный класс'
  'отрицательный класс'
  'множество $\\{-1, 1\\}$'
  'признак $x_i$'
  'пространство $\\mathbb{R}^D$'
  'линейная модель'


In [18]:
show_edge_alignments(clustered, gt_graph, metrics, top_k=TOP_K)

Matched edge pairs: 2 / min(2, 51)=2

Top 2 matched (by quality q):
  [q=0.918]  SVM —[выдавать]→ мы
           ↔  вектор —[принадлежит]→ пространство $\mathbb{R}^D$
  [q=0.911]  мы —[выбирать]→ SVM
           ↔  признак $x_i$ —[является]→ вектор

Unmatched GT edges (49):
  бинарная классификация —[является частным случаем]→ задача классификации
  бинарная классификация —[обобщается до]→ классификация на $K$ классов
  таргет $y$ —[кодирует принадлежность к]→ положительный класс
  таргет $y$ —[кодирует принадлежность к]→ отрицательный класс
  таргет $y$ —[принимает значения из]→ множество $\{-1, 1\}$
  линейная модель —[параметризуется]→ веса $w$
  линейная модель —[задаёт]→ разделяющая плоскость
  разделяющая плоскость —[разделяет классы в]→ бинарная классификация
  выборка —[может обладать свойством]→ линейная разделимость
  предсказание $\hat{y}$ —[определяется как]→ $\hat{y}=\operatorname{sign}\langle w, x_i\rangle$


In [19]:
save_json(metrics.summary(), out / 'benchmark_metrics.json')
print(f'Saved benchmark_metrics.json to {out}/')

Saved benchmark_metrics.json to /home/platoon/graph/semantic-graph/exps/FINAL_EXPS/phase1a_embedders/exp_e3_e5_large/output/


## Benchmark — multi-method / multi-threshold sweep

In [20]:
is_multi = config.clustering.multi_method or config.clustering.is_multi_threshold

if not is_multi:
    print('Skipped: multi-method / multi-threshold not enabled in config')
    multi_metrics = None
else:
    total = sum(len(mr.graphs) for mr in multi.methods.values())
    print(f'Evaluating {total} configurations...')
    multi_metrics = evaluate_multi_graph(
        multi=multi,
        gt=gt_graph,
        source_text=source_text,
        embedder=embedder,
        tau_node=TAU_NODE,
        tau_edge=TAU_EDGE,
        beta=BETA,
        node_weight=NODE_WEIGHT,
        edge_weight=EDGE_WEIGHT,
        node_window=NODE_WINDOW,
        edge_window=EDGE_WINDOW,
    )
    print(f'Done: {sum(len(v) for v in multi_metrics.values())} variants evaluated')

Evaluating 23 configurations...


Batches: 100%|██████████| 1/1 [00:00<00:00, 111.66it/s]

Done: 23 variants evaluated


In [21]:
if multi_metrics:
    print_multi_metrics(multi_metrics, sort_by='graph_score')

method        param                  pred_n pred_e  matched_n  matched_e    P_n    R_n    F_n    P_e    R_e    F_e   graph
--------------------------------------------------------------------------------------------------------------------------
agglomerative 0.322                      47     58         47         51  0.718  0.614  0.662  0.666  0.758  0.709  0.6808
agglomerative 0.394                      41     53         41         51  0.726  0.541  0.620  0.716  0.744  0.730  0.6638
agglomerative 0.250                      71     81         55         51  0.571  0.738  0.644  0.493  0.783  0.605  0.6284
agglomerative 0.467                      26     44         26         44  0.776  0.367  0.498  0.738  0.637  0.684  0.5725
agglomerative 0.539                      14     21         14         21  0.865  0.220  0.351  0.838  0.345  0.489  0.4062
agglomerative 0.611                       2      2          2          2  0.899  0.033  0.063  0.914  0.036  0.069  0.0655
agglomerative 0.

In [22]:
if multi_metrics:
    method, param, best_m = best_variant(multi_metrics, by='graph_score')
    best_graph = multi.methods[method].graphs[param]
    print(f'Best variant: method={method}, param={param}')
    print(f'  graph: {len(best_graph.nodes)} nodes, {len(best_graph.edges)} edges')
    print()
    print_metrics(best_m)

Best variant: method=kmeans, param=k=55
  graph: 55 nodes, 64 edges

GraphScore = 0.6·NodeF1 + 0.4·EdgeF1  =  0.6944

Nodes (pred=55, gt=55, matched=55, tau=0.6, beta=1):
  TP(soft)  = 38.5972
  precision = 0.7018
  recall    = 0.7018
  F1        = 0.7018
Edges (pred=64, gt=51, matched=51, tau=0.6, beta=1):
  TP(soft)  = 39.2993
  precision = 0.6141
  recall    = 0.7706
  F1        = 0.6835


In [23]:
if multi_metrics:
    save_json(multi_metrics_to_dict(multi_metrics), out / 'benchmark_metrics_multi.json')
    print(f'Saved benchmark_metrics_multi.json to {out}/')

Saved benchmark_metrics_multi.json to /home/platoon/graph/semantic-graph/exps/FINAL_EXPS/phase1a_embedders/exp_e3_e5_large/output/
